# FIONA Treatment Example

This notebook tests `shrecc.treatment_fiona` from the hourly TYNDP pickle files. It builds `Z_gross`, computes the hourly consumption mix, and checks that each consumer-country/hour mix sums to one.

In [17]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

cwd = Path.cwd().resolve()

for candidate in [cwd, *cwd.parents]:
    if (candidate / "shrecc" / "treatment_fiona.py").exists():
        project_root = candidate
        break
    if (candidate / "shrecc" / "shrecc" / "treatment_fiona.py").exists():
        project_root = candidate / "shrecc"
        break
else:
    raise RuntimeError("Could not find the local shrecc package root")

module_path = project_root / "shrecc" / "treatment_fiona.py"
spec = importlib.util.spec_from_file_location("treatment_fiona", module_path)

if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load {module_path}")

treatment_fiona = importlib.util.module_from_spec(spec)
spec.loader.exec_module(treatment_fiona)

build_z_gross_from_tyndp_excel = treatment_fiona.build_z_gross_from_tyndp_excel
build_z_gross_from_tyndp_pickles = treatment_fiona.build_z_gross_from_tyndp_pickles
consumption_mix_from_tyndp_excel = treatment_fiona.consumption_mix_from_tyndp_excel
consumption_mix_from_tyndp_pickles = treatment_fiona.consumption_mix_from_tyndp_pickles
consumption_mix_from_z_gross = treatment_fiona.consumption_mix_from_z_gross

In [18]:
data_dir = project_root / "data" / "tyndp"

tyndp_workbook = data_dir / "MMStandardOutputFile_DE2050_Plexos_CY2009_v11_SoS.xlsb"
production_pickle = data_dir / "DE2050_CY2009_prod.pkl"
trade_pickle = data_dir / "DE2050_CY2009_trade.pkl"
technology_mapping = data_dir / "tyndp_ei_mapping.xlsx"
country_mapping = data_dir / "tyndp_country_mapping.xlsx"

for path in [tyndp_workbook, production_pickle, trade_pickle, technology_mapping, country_mapping]:
    assert path.exists(), path

## Build `Z_gross`

`Z_gross` is the gross hourly supply table with production and trade flows in one column structure.

In [19]:
Z_gross = build_z_gross_from_tyndp_pickles(
    production_pickle=production_pickle,
    trade_pickle=trade_pickle,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    verbose=True,
)

Z_gross.shape, Z_gross.columns.names, Z_gross.index[:3]

2026-05-11 15:03:02 Loading TYNDP pickle files
2026-05-11 15:03:03 Parsing hourly datetime index
2026-05-11 15:03:03 Loading technology and country mappings
2026-05-11 15:03:03 Aggregating hourly production by country and technology


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:531: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  production = production.groupby(


2026-05-11 15:03:03 Aggregating hourly trade to positive directional country flows


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:233: UserWarning: Dropping trade columns without country-pair mappings: AT00-AT00EV, AT00-AT00RETE, AT00EV-AT00, AT00EV-AT00RETE, AT00RETE-AT00EV, BE00-BE00EV, BE00-BE00RETE, BE00EV-BE00, BE00EV-BE00RETE, BE00RETE-BE00EV
  trade_directional = _aggregate_tyndp_trade(trade, connections)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:593: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  trade = trade.groupby(level=["Country from", "Country to"], axis=1).sum()


2026-05-11 15:03:04 Combining production and trade into Z_gross
2026-05-11 15:03:04 Built Z_gross with shape (8736, 485)


((8736, 485),
 FrozenList(['type', 'country from', 'country to', 'source']),
 DatetimeIndex(['2050-01-01 00:00:00', '2050-01-01 01:00:00',
                '2050-01-01 02:00:00'],
               dtype='datetime64[ns]', freq=None))

In [20]:
assert len(Z_gross) == 8736
assert Z_gross.columns.names == ["type", "country from", "country to", "source"]
assert {"production mix", "trade"}.issubset(Z_gross.columns.get_level_values("type"))

Z_gross.head(2)

type                   production mix                                     \
country from                       AL                                      
country to                         AL                                      
source              Gas CCGT new [MW] Gas CCGT old 2 [MW] Reservoir [MW]   
2050-01-01 00:00:00               110                   0           1904   
2050-01-01 01:00:00               100                   0           1904   

type                                                             \
country from                                                      
country to                                                        
source              Run-of-River [MW] Solar (Photovoltaic) [MW]   
2050-01-01 00:00:00               193                         0   
2050-01-01 01:00:00               193                         0   

type                                                      \
country from                                          AT   
country to                                            AT   
source              Wind Onshore [MW] Hydrogen CCGT [MW]   
2050-01-01 00:00:00                40               2934   
2050-01-01 01:00:00                43               2934   

type                                                                 \
country from                                                          
country to                                                            
source              Others non-renewable [MW] Others renewable [MW]   
2050-01-01 00:00:00                       464                   502   
2050-01-01 01:00:00                       464                   502   

type                                                           ... trade  \
country from                                                   ...    SI   
country to                                                     ...    IT   
source              Pump Storage - Closed Loop (turbine) [MW]  ... trade   
2050-01-01 00:00:00                                        31  ...     0   
2050-01-01 01:00:00                                        31  ...     0   

type                                                                       
country from           SK                UK                                
country to             CZ    HU    PL    BE    DE    DK    FR    IE    NL  
source              trade trade trade trade trade trade trade trade trade  
2050-01-01 00:00:00  2600     0     0  1000  2800     0     0     0  1000  
2050-01-01 01:00:00  2600     0     0  1000  2800     0     0     0  1000  

[2 rows x 485 columns]

## Compute Consumption Mix From `Z_gross`

The debug output includes the direct production/trade coefficient matrices used for the conservation checks.

In [21]:
consumption_mix_xr, debug = consumption_mix_from_z_gross(
    Z_gross,
    check=True,
    return_debug=True,
    verbose=True,
)

consumption_mix_xr

2026-05-11 15:03:13 Building domestic production and trade denominator matrices


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:700: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(["country from", "country to"], axis=1)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:720: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  country_total = Z_trade.groupby("country to", axis=1).sum()


2026-05-11 15:03:13 Calculating direct production and trade shares
2026-05-11 15:03:13 Using 35 countries and 32 technologies
2026-05-11 15:03:13 Reshaping production shares
2026-05-11 15:03:14 Reshaping trade shares
2026-05-11 15:03:14 Checking direct shares sum to one
2026-05-11 15:03:14 Solving hourly trade system
2026-05-11 15:03:21 Checking final consumption mix sums to one
2026-05-11 15:03:21 Building xarray DataArray
2026-05-11 15:03:38 Built consumption_mix_xr with shape (8736, 35, 35, 32)


<xarray.DataArray 'consumption_mix' (time: 8736, consumer_country: 35,
                                     source_country: 35, technology: 32)> Size: 3GB
array([[[[0.00000000e+00, 2.41652021e-02, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 8.78734622e-03],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 1.61824034e-04, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 5.88451033e-05],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.41976107e-01],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.57459439e-03],
...
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 1.79452669e-02],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 1.99220275e-04, 1.38429298e-04]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 5.15654365e-01, 3.58305258e-01]]]],
      shape=(8736, 35, 35, 32))
Coordinates:
  * time              (time) datetime64[ns] 70kB 2050-01-01 ... 2050-12-30T23...
  * consumer_country  (consumer_country) object 280B 'AL' 'AT' ... 'SK' 'UK'
  * source_country    (source_country) object 280B 'AL' 'AT' 'BA' ... 'SK' 'UK'
  * technology        (technology) object 256B 'Gas CCGT CCS [MW]' ... 'Wind ...

In [22]:
direct_total = debug["direct_total"]

direct_total.min(), direct_total.max()

(np.float64(0.9999999999999997), np.float64(1.0000000000000002))

In [23]:
mix_total = consumption_mix_xr.sum(["source_country", "technology"])

float(mix_total.min()), float(mix_total.max())

(0.9999999999999997, 1.0000000000000004)

In [24]:
np.testing.assert_allclose(direct_total, 1, atol=1e-8)
np.testing.assert_allclose(mix_total, 1, atol=1e-8)

print("All conservation checks passed.")

All conservation checks passed.


## One-Step Pipeline

The convenience wrapper should give the same result as building `Z_gross` explicitly first.

In [25]:
consumption_mix_direct = consumption_mix_from_tyndp_pickles(
    production_pickle=production_pickle,
    trade_pickle=trade_pickle,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    check=True,
    verbose=True,
)

xr.testing.assert_allclose(consumption_mix_direct, consumption_mix_xr)

consumption_mix_direct.shape

2026-05-11 15:03:47 Loading TYNDP pickle files
2026-05-11 15:03:48 Parsing hourly datetime index
2026-05-11 15:03:48 Loading technology and country mappings
2026-05-11 15:03:48 Aggregating hourly production by country and technology


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:531: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  production = production.groupby(


2026-05-11 15:03:49 Aggregating hourly trade to positive directional country flows


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:233: UserWarning: Dropping trade columns without country-pair mappings: AT00-AT00EV, AT00-AT00RETE, AT00EV-AT00, AT00EV-AT00RETE, AT00RETE-AT00EV, BE00-BE00EV, BE00-BE00RETE, BE00EV-BE00, BE00EV-BE00RETE, BE00RETE-BE00EV
  trade_directional = _aggregate_tyndp_trade(trade, connections)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:593: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  trade = trade.groupby(level=["Country from", "Country to"], axis=1).sum()


2026-05-11 15:03:49 Combining production and trade into Z_gross
2026-05-11 15:03:49 Built Z_gross with shape (8736, 485)
2026-05-11 15:03:49 Building domestic production and trade denominator matrices


D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:700: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(["country from", "country to"], axis=1)
D:\shrecc_project\shrecc\shrecc\treatment_fiona.py:720: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  country_total = Z_trade.groupby("country to", axis=1).sum()


2026-05-11 15:03:49 Calculating direct production and trade shares
2026-05-11 15:03:49 Using 35 countries and 32 technologies
2026-05-11 15:03:49 Reshaping production shares
2026-05-11 15:03:51 Reshaping trade shares
2026-05-11 15:03:52 Checking direct shares sum to one
2026-05-11 15:03:52 Solving hourly trade system
2026-05-11 15:03:58 Checking final consumption mix sums to one
2026-05-11 15:03:58 Building xarray DataArray
2026-05-11 15:04:12 Built consumption_mix_xr with shape (8736, 35, 35, 32)


(8736, 35, 35, 32)

## Excel/XLSB Input

If the pickle files are not available, the same `Z_gross` table can be built directly from the original TYNDP workbook. Reading the `.xlsb` file is much slower than loading the pickle files, so this section is mainly a from-scratch check.

In [ ]:
Z_gross_from_excel = build_z_gross_from_tyndp_excel(
    excel_file=tyndp_workbook,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    verbose=True,
)

Z_gross_from_excel.shape, Z_gross_from_excel.columns.names, Z_gross_from_excel.index[:3]

In [ ]:
pd.testing.assert_frame_equal(Z_gross_from_excel, Z_gross)

print("Excel-built Z_gross matches pickle-built Z_gross.")

The one-step Excel wrapper reads the workbook, builds `Z_gross`, and computes `consumption_mix_xr` in one call. This repeats the long workbook read and matrix solve, so run it when you specifically want to test the full Excel path.

In [ ]:
consumption_mix_from_excel = consumption_mix_from_tyndp_excel(
    excel_file=tyndp_workbook,
    technology_mapping=technology_mapping,
    country_mapping=country_mapping,
    check=True,
    verbose=True,
)

xr.testing.assert_allclose(consumption_mix_from_excel, consumption_mix_xr)

consumption_mix_from_excel.shape

## Inspect One Country And Hour

In [ ]:
consumer_country = "DE"
time = consumption_mix_xr.time.values[0]

top_sources = (
    consumption_mix_xr.sel(consumer_country=consumer_country, time=time)
    .to_dataframe(name="share")
    .sort_values("share", ascending=False)
    .head(20)
)

top_sources